# Lyapunov Spectra Smoke Campaign: `N20x40`

Colab/A100 notebook for the `colab_lyapunov` smoke campaign. It runs canonical GPU Markov dynamics through `classA_U1FGTN_gpu.run_markov_circuit(...)` and streams Lyapunov spectra, real-space Chern number, and unit-cell charge at every post-update cycle.


In [ ]:

from __future__ import annotations

import json
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_lyapunov'


def is_bundle_root(path: Path) -> bool:
    return (
        (path / 'src' / 'classA_U1FGTN_gpu.py').exists()
        and (path / 'src' / 'lyapunov_observables_gpu.py').exists()
    )


candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with required src files')

PROJECT_ROOT = BUNDLE_ROOT.parent
BUNDLE_SRC_DIR = BUNDLE_ROOT / 'src'
if str(BUNDLE_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_SRC_DIR))

print(f'[setup] bundle root: {BUNDLE_ROOT}')
print(f'[setup] project root: {PROJECT_ROOT}')


In [ ]:

from contextlib import contextmanager
import inspect
import threading
import time
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from tqdm.auto import tqdm

from classA_U1FGTN_gpu import classA_U1FGTN_gpu
import lyapunov_observables_gpu as lyap_helper
from lyapunov_observables_gpu import (
    CANONICAL_ENTRY_POINT,
    CHANNEL_ORDER,
    HELPER_VERSION,
    LyapunovSmokeObserver,
    case_output_dir,
    config_id,
    domain_wall_metadata,
    expected_samples_for_config,
    rel_to_root,
    run_internal_checks,
    save_dataframe_atomic_csv,
    save_npz_atomic,
    write_json_atomic,
)

HEARTBEAT_INTERVAL_S = 60.0


@contextmanager
def progress_heartbeat(progress_bar, config_label: str, *, interval_s: float = HEARTBEAT_INTERVAL_S):
    if interval_s <= 0:
        raise ValueError('heartbeat interval must be positive')
    stop_event = threading.Event()
    started = time.perf_counter()

    def emit() -> None:
        while not stop_event.wait(interval_s):
            elapsed_s = time.perf_counter() - started
            progress_bar.set_postfix_str(f'{config_label} elapsed={elapsed_s:.0f}s', refresh=True)
            progress_bar.write(f'[heartbeat] {config_label} still running; elapsed={elapsed_s:.0f}s')

    worker = threading.Thread(target=emit, name='colab-progress-heartbeat', daemon=True)
    worker.start()
    try:
        yield
    finally:
        stop_event.set()
        worker.join()


EXPECTED_HELPER_VERSION = 'lyapunov_observables_gpu_v1'
EXPECTED_CHANNEL_ORDER = ('Ap', 'Am', 'Bp', 'Bm')
EXPECTED_GPU_SOURCE = (BUNDLE_SRC_DIR / 'classA_U1FGTN_gpu.py').resolve()
HELPER_MODULE_PATH = Path(inspect.getsourcefile(lyap_helper) or lyap_helper.__file__).resolve()
GPU_MODULE_PATH = Path(inspect.getsourcefile(classA_U1FGTN_gpu) or '').resolve()
RUN_SIGNATURE = inspect.signature(classA_U1FGTN_gpu.run_markov_circuit)

if HELPER_VERSION != EXPECTED_HELPER_VERSION:
    raise RuntimeError(f'Wrong helper version: expected {EXPECTED_HELPER_VERSION}, got {HELPER_VERSION}')
if tuple(CHANNEL_ORDER) != EXPECTED_CHANNEL_ORDER:
    raise RuntimeError(f'Wrong channel order metadata: {CHANNEL_ORDER}')
if GPU_MODULE_PATH != EXPECTED_GPU_SOURCE:
    raise RuntimeError(f'Loaded classA_U1FGTN_gpu from {GPU_MODULE_PATH}, expected {EXPECTED_GPU_SOURCE}')
if 'lyapunov_observer' not in RUN_SIGNATURE.parameters:
    raise RuntimeError(f'Stale classA_U1FGTN_gpu.run_markov_circuit signature: {RUN_SIGNATURE}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

torch.set_grad_enabled(False)
run_internal_checks()

total_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'[setup] helper={HELPER_VERSION}')
print(f'[setup] helper source: {HELPER_MODULE_PATH}')
print(f'[setup] gpu source: {GPU_MODULE_PATH}')
print(f'[setup] run signature: {RUN_SIGNATURE}')
print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')
print(f'[setup] total gpu memory: {total_gib:.2f} GiB')
if total_gib < 35.0:
    print('[warn] This notebook is sized for an A100 40GB runtime.')


## Campaign Configuration

This smoke campaign uses `cycles=20` and requested `samples=10`. The canonical GPU driver forces post-selection to one actual trajectory.


In [ ]:

NX = 20
NY = 40
CYCLES = 20
REQUESTED_SAMPLES = 10
NSHELL = 1
ALPHA_1 = 1.0
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
TRIAL_ORBITALS = 'X'
INIT_MODE = 'default'
N_A = 0.5
MEAS_SLAB_ONLY = False  # Preserve this legacy full-lattice tangent-spectrum campaign.
LYAPUNOV_NVEC = 2 * NX * NY
BATCH_SIZE_PERFECT = 1
BATCH_SIZE_POSTSELECT = 1
OVERWRITE = True
SKIP_COMPLETED = True
RUN_TINY_INTERNAL_CHECK = False

DW_CONFIGS = [
    {'DW': True, 'dw_truncation': True, 'alpha_2': 30.0, 'config_label': 'DW1_dwtrunc1_a2-30'},
    {'DW': False, 'dw_truncation': False, 'alpha_2': 1.0, 'config_label': 'DW0_dwtrunc0_a2-1'},
]

PROTOCOL_CONFIGS = [
    {
        'protocol': 'perfect_correction',
        'postselect': False,
        'perfect_correction': True,
        'samples': REQUESTED_SAMPLES,
        'batch_size': BATCH_SIZE_PERFECT,
    },
    {
        'protocol': 'postselect',
        'postselect': True,
        'perfect_correction': False,
        'samples': REQUESTED_SAMPLES,
        'batch_size': BATCH_SIZE_POSTSELECT,
    },
]

CONFIGS = [
    {
        'Nx': NX,
        'Ny': NY,
        'cycles': CYCLES,
        'nshell': NSHELL,
        'alpha_1': ALPHA_1,
        'init_mode': INIT_MODE,
        **dw_cfg,
        **protocol_cfg,
    }
    for dw_cfg in DW_CONFIGS
    for protocol_cfg in PROTOCOL_CONFIGS
]

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'lyapunov_spectra'
CAMPAIGN_ID = 'N20x40_nsh1_a1-1_S10_cycles20_smoke'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
CAMPAIGN_ROOT = OUTPUT_ROOT / 'campaigns' / CAMPAIGN_ID
RUNS_ROOT = CAMPAIGN_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

config_rows = []
for cfg in CONFIGS:
    config_rows.append({
        'config_id': config_id(cfg),
        'Nx': cfg['Nx'],
        'Ny': cfg['Ny'],
        'cycles': cfg['cycles'],
        'requested_samples': cfg['samples'],
        'actual_samples': expected_samples_for_config(cfg),
        'protocol': cfg['protocol'],
        'DW': cfg['DW'],
        'dw_truncation': cfg['dw_truncation'],
        'alpha_1': cfg['alpha_1'],
        'alpha_2': cfg['alpha_2'],
        'n_vec': LYAPUNOV_NVEC,
        'batch_size': cfg['batch_size'],
    })
config_df = pd.DataFrame(config_rows)
display(Markdown('### Smoke campaign config'))
display(config_df)
print(json.dumps({'campaign_id': CAMPAIGN_ID, 'channel_order': list(CHANNEL_ORDER), 'configs': CONFIGS}, indent=2))


In [ ]:

def file_size_bytes(path: Path | None) -> int:
    return int(Path(path).stat().st_size) if path is not None and Path(path).exists() else 0


def assert_overwrite_allowed(path: Path) -> None:
    if path.exists() and not OVERWRITE:
        raise FileExistsError(f'Path already exists and OVERWRITE=False: {path}')


def expected_output_paths(out_dir: Path) -> dict[str, Path]:
    return {
        'spectra_path': out_dir / 'lyapunov_spectra.npz',
        'min_abs_vector_path': out_dir / 'lyapunov_min_abs_vector.npz',
        'chern_path': out_dir / 'real_space_chern.npz',
        'charge_path': out_dir / 'local_charge_cell.npz',
        'metrics_path': out_dir / 'scalar_metrics.csv',
        'summary_path': out_dir / 'run_summary.json',
        'latest_path': out_dir / 'latest_run.json',
    }


def completed_result_if_available(cfg: dict[str, Any], out_dir: Path) -> dict[str, Any] | None:
    latest_path = out_dir / 'latest_run.json'
    if not SKIP_COMPLETED or not latest_path.exists():
        return None
    payload = json.loads(latest_path.read_text())
    if payload.get('status') == 'completed' and payload.get('config_id') == config_id(cfg):
        print(f"[skip] completed {config_id(cfg)}")
        return payload['run_index_row']
    return None


def run_one_config(cfg: dict[str, Any]) -> dict[str, Any]:
    out_dir = case_output_dir(RUNS_ROOT, cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    completed = completed_result_if_available(cfg, out_dir)
    if completed is not None:
        return completed

    expected_samples = expected_samples_for_config(cfg)
    print(
        f"[config] {config_id(cfg)} cycles={cfg['cycles']} requested_samples={cfg['samples']} "
        f"actual_samples={expected_samples} out={out_dir}"
    )

    model = classA_U1FGTN_gpu(
        Nx=int(cfg['Nx']),
        Ny=int(cfg['Ny']),
        DW=bool(cfg['DW']),
        nshell=int(cfg['nshell']),
        filling_frac=0.5,
        alpha_1=float(cfg['alpha_1']),
        alpha_2=float(cfg['alpha_2']),
        trial_orbitals=TRIAL_ORBITALS,
        dw_truncation=bool(cfg['dw_truncation']),
        triv_region_local_mode=False,
        device=DEVICE,
        dtype=DTYPE,
        backend=BACKEND,
    )
    dw_meta = domain_wall_metadata(model)
    observer = LyapunovSmokeObserver(
        nx=int(cfg['Nx']),
        ny=int(cfg['Ny']),
        cycles=int(cfg['cycles']),
        samples_expected=expected_samples,
        n_vec=LYAPUNOV_NVEC,
        protocol=str(cfg['protocol']),
        chern_metadata=dw_meta,
    )

    run_start = time.perf_counter()
    result = model.run_markov_circuit(
        G_history=False,
        progress=False,
        cycles=int(cfg['cycles']),
        postselect=bool(cfg['postselect']),
        perfect_correction=bool(cfg['perfect_correction']),
        samples=int(cfg['samples']),
        init_mode=str(cfg['init_mode']),
        save=False,
        save_init=False,
        n_a=N_A,
        sequence=SEQUENCE,
        meas_slab_only=MEAS_SLAB_ONLY,
        batch_size=int(cfg['batch_size']),
        return_data=False,
        lyapunov_observer=observer,
        lyapunov_nvec=LYAPUNOV_NVEC,
    )
    elapsed_s = float(time.perf_counter() - run_start)
    actual_samples = int(result['samples'])
    if actual_samples != expected_samples:
        raise RuntimeError(f"Expected {expected_samples} actual samples, got {actual_samples}")
    observer.finalize(actual_samples=actual_samples)

    run_payload = {
        **cfg,
        'config_id': config_id(cfg),
        'actual_samples': actual_samples,
        'lyapunov_nvec': LYAPUNOV_NVEC,
        'channel_order': list(CHANNEL_ORDER),
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'helper_version': HELPER_VERSION,
        'dw_metadata': dw_meta,
        'elapsed_s': elapsed_s,
        'batch_size': int(result['batch_size']),
        'batch_size_mode': result['batch_size_mode'],
        'meas_slab_only': MEAS_SLAB_ONLY,
    }

    paths = expected_output_paths(out_dir)
    for path in paths.values():
        assert_overwrite_allowed(path)

    metrics_df = observer.metrics_dataframe().sort_values(['sample_index', 'cycle']).reset_index(drop=True)
    save_npz_atomic(paths['spectra_path'], **observer.spectra_npz_payload(config=run_payload))
    save_npz_atomic(paths['min_abs_vector_path'], **observer.min_abs_vector_npz_payload(config=run_payload))
    save_npz_atomic(paths['chern_path'], **observer.chern_npz_payload(config=run_payload))
    save_npz_atomic(paths['charge_path'], **observer.charge_npz_payload(config=run_payload))
    save_dataframe_atomic_csv(metrics_df, paths['metrics_path'])

    file_sizes = {key.replace('_path', ''): file_size_bytes(path) for key, path in paths.items()}
    run_index_row = {
        'config_id': config_id(cfg),
        'protocol': cfg['protocol'],
        'DW': bool(cfg['DW']),
        'dw_truncation': bool(cfg['dw_truncation']),
        'alpha_2': float(cfg['alpha_2']),
        'samples_requested': int(cfg['samples']),
        'samples_actual': actual_samples,
        'cycles': int(cfg['cycles']),
        'lyapunov_nvec': LYAPUNOV_NVEC,
        'run_dir_relative': rel_to_root(out_dir, GPU_DATA_ROOT),
        'summary_relative': rel_to_root(paths['summary_path'], GPU_DATA_ROOT),
        'saved_total_bytes': int(sum(file_sizes.values())),
    }
    summary = {
        **run_payload,
        'status': 'completed',
        'run_dir': str(out_dir),
        'run_dir_relative': rel_to_root(out_dir, GPU_DATA_ROOT),
        'paths_relative': {key: rel_to_root(path, GPU_DATA_ROOT) for key, path in paths.items()},
        'file_sizes_bytes': file_sizes,
        'metric_row_count': int(len(metrics_df)),
        'run_index_row': run_index_row,
    }
    write_json_atomic(paths['summary_path'], summary)
    write_json_atomic(paths['latest_path'], summary)
    return run_index_row


In [ ]:
if RUN_TINY_INTERNAL_CHECK:
    tiny_cfg = {
        'Nx': 4,
        'Ny': 4,
        'cycles': 2,
        'nshell': 1,
        'alpha_1': 1.0,
        'alpha_2': 1.0,
        'DW': False,
        'dw_truncation': False,
        'init_mode': INIT_MODE,
        'protocol': 'perfect_correction',
        'postselect': False,
        'perfect_correction': True,
        'samples': 2,
        'batch_size': 1,
    }
    old_nvec = LYAPUNOV_NVEC
    try:
        LYAPUNOV_NVEC = 2 * tiny_cfg['Nx'] * tiny_cfg['Ny']
        tiny_result = run_one_config(tiny_cfg)
        tiny_out = case_output_dir(RUNS_ROOT, tiny_cfg)
        with np.load(tiny_out / 'lyapunov_spectra.npz') as data:
            spectra = data['lyapunov_spectra']
        with np.load(tiny_out / 'real_space_chern.npz') as data:
            chern = data['real_space_chern']
        with np.load(tiny_out / 'local_charge_cell.npz') as data:
            charge = data['local_charge_cell']
        with np.load(tiny_out / 'lyapunov_min_abs_vector.npz') as data:
            min_abs_vector = data['lyapunov_min_abs_vector']
            min_abs_value = data['lyapunov_min_abs_value']
            min_abs_index = data['lyapunov_min_abs_index']
        expected_spectra_shape = (2, 2, LYAPUNOV_NVEC)
        expected_chern_shape = (2, 2)
        expected_charge_shape = (2, 2, 4, 4)
        expected_vector_shape = (2, LYAPUNOV_NVEC)
        if spectra.shape != expected_spectra_shape:
            raise RuntimeError(f'tiny spectra shape {spectra.shape} != {expected_spectra_shape}')
        if chern.shape != expected_chern_shape:
            raise RuntimeError(f'tiny Chern shape {chern.shape} != {expected_chern_shape}')
        if charge.shape != expected_charge_shape:
            raise RuntimeError(f'tiny charge shape {charge.shape} != {expected_charge_shape}')
        if min_abs_vector.shape != expected_vector_shape:
            raise RuntimeError(f'tiny min-abs vector shape {min_abs_vector.shape} != {expected_vector_shape}')
        if min_abs_value.shape != (2,):
            raise RuntimeError(f'tiny min-abs value shape {min_abs_value.shape} != {(2,)}')
        if min_abs_index.shape != (2,):
            raise RuntimeError(f'tiny min-abs index shape {min_abs_index.shape} != {(2,)}')
        if not (
            np.isfinite(spectra).all()
            and np.isfinite(chern).all()
            and np.isfinite(charge).all()
            and np.isfinite(min_abs_vector).all()
            and np.isfinite(min_abs_value).all()
        ):
            raise FloatingPointError('tiny run produced non-finite output')
        print('[tiny] passed', tiny_result)
    finally:
        LYAPUNOV_NVEC = old_nvec
else:
    print('[tiny] skipped; set RUN_TINY_INTERNAL_CHECK=True for manual tiny validation')


In [ ]:

all_results = []
campaign_start = time.perf_counter()
campaign_bar = tqdm(total=len(CONFIGS), desc='Lyapunov smoke configurations', unit='config', leave=True, dynamic_ncols=True)
try:
    for cfg in CONFIGS:
        label = config_id(cfg)
        campaign_bar.set_postfix_str(f'{label} elapsed=0s', refresh=True)
        with progress_heartbeat(campaign_bar, label):
            all_results.append(run_one_config(cfg))
        campaign_bar.update(1)
finally:
    campaign_bar.close()
elapsed_s = float(time.perf_counter() - campaign_start)

run_index = pd.DataFrame(all_results).sort_values(['DW', 'dw_truncation', 'alpha_2', 'protocol']).reset_index(drop=True)
run_index_path = CAMPAIGN_ROOT / 'run_index.csv'
assert_overwrite_allowed(run_index_path)
save_dataframe_atomic_csv(run_index, run_index_path)

campaign_manifest = {
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'helper_version': HELPER_VERSION,
    'channel_order': list(CHANNEL_ORDER),
    'bundle_root': str(BUNDLE_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'run_index_relative': rel_to_root(run_index_path, GPU_DATA_ROOT),
    'configs': CONFIGS,
    'results': all_results,
    'elapsed_s': elapsed_s,
}
write_json_atomic(CAMPAIGN_ROOT / 'campaign_manifest.json', campaign_manifest)
write_json_atomic(CAMPAIGN_ROOT / 'latest_campaign.json', campaign_manifest)

display(Markdown('### Run index'))
display(run_index)
print(json.dumps(campaign_manifest, indent=2))


In [ ]:

from IPython.display import Javascript, display

try:
    display(Javascript('google.colab.kernel.disconnect()'))
except Exception as exc:
    print(f'[disconnect] skipped: {exc}')
